In [3]:
pip install opencv-python face_recognition
numpy pandas

SyntaxError: invalid syntax (2137473.py, line 1)

In [4]:
import cv2
import face_recognition
import numpy as np
import os
from datetime import datetime

IMAGE_DIR = 'images'
known_encodings = []
known_names = []

for filename in os.listdir(IMAGE_DIR):
    if filename.lower().endswith(('.png', '.jpg', '.jpeg')):
        img_path = os.path.join(IMAGE_DIR, filename)
        image = face_recognition.load_image_file(img_path)
        encodings = face_recognition.face_encodings(image)
        
        if encodings:
            known_encodings.append(encodings[0])
            known_names.append(os.path.splitext(filename)[0].replace('_', ' ').title())

def mark_attendance(name):
    filename = 'attendance.csv'
    today_str = datetime.now().strftime('%Y-%m-%d')
    time_str = datetime.now().strftime('%H:%M:%S')

    if not os.path.exists(filename):
        with open(filename, 'w') as f:
            f.write('Name,Date,Time\n')

    with open(filename, 'r') as f:
        existing_records = [line.strip().split(',') for line in f.readlines()]
        for record in existing_records:
            if len(record) >= 2 and record[0] == name and record[1] == today_str:
                return

    with open(filename, 'a') as f:
        f.write(f'{name},{today_str},{time_str}\n')
        print(f"Logged attendance for {name} at {time_str}")

cap = cv2.VideoCapture(0)

print("Starting camera... Press 'q' to quit.")

while True:
    success, frame = cap.read()
    if not success:
        break
    small_frame = cv2.resize(frame, (0, 0), fx=0.25, fy=0.25)
    rgb_small = cv2.cvtColor(small_frame, cv2.COLOR_BGR2RGB)

    face_locations = face_recognition.face_locations(rgb_small)
    face_encodings = face_recognition.face_encodings(rgb_small, face_locations)

    for encode_face, face_loc in zip(face_encodings, face_locations):
        matches = face_recognition.compare_faces(known_encodings, encode_face, tolerance=0.5)
        face_distances = face_recognition.face_distance(known_encodings, encode_face)

        name = "Unknown"

        if len(face_distances) > 0:
            best_match_index = np.argmin(face_distances)
            if matches[best_match_index]:
                name = known_names[best_match_index]
                mark_attendance(name)

        top, right, bottom, left = [coord * 4 for coord in face_loc]

        box_color = (0, 255, 0) if name != "Unknown" else (0, 0, 255)
        cv2.rectangle(frame, (left, top), (right, bottom), box_color, 2)
        cv2.rectangle(frame, (left, bottom - 35), (right, bottom), box_color, cv2.FILLED)
        cv2.putText(frame, name, (left + 6, bottom - 8), cv2.FONT_HERSHEY_DUPLEX, 0.8, (255, 255, 255), 1)

    cv2.imshow('Face Attendance System', frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

ModuleNotFoundError: No module named 'cv2'